# Text branch v2: Transformer pipeline for multimodal emotion detection

Continues `DL_text_detection2.ipynb` (BiLSTM baseline on GoEmotions). This notebook is self-contained.

```
text -> RoBERTa encoder -> mean pooling -> 768-d embedding ----+--> 28-way GoEmotions head -> per-emotion thresholds
                                                               |         |
                                                               |         +--> Ekman-7 mapping (anger, disgust, fear, joy, sadness, surprise, neutral)
                                                               |
                                                               +--> exported for the multimodal fusion model (with audio + video features)
```

**Stages**
1. Data and tokenization (subword tokenizer, dynamic padding)
2. Model: pretrained encoder + mean pooling + multi-label head (also returns the embedding for fusion)
3. Imbalance-aware loss (`bce`, `bce_posweight`, `focal`)
4. Training: AMP, warmup + linear decay, layer-group learning rates, early stopping
5. Threshold tuning on validation only (global and per-emotion with a small-class fallback)
6. Evaluation: 28 emotions and Ekman-7, per-class report, error analysis
7. Saving artifacts and a reusable `TextEmotionPredictor`
8. Fusion-ready feature export and a missing-modality-safe fusion module

## 0. Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn seaborn

In [ ]:
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    DataCollatorWithPadding, get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, hamming_loss, classification_report
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
class CFG:
    model_name       = "roberta-base"    # "distilroberta-base" is ~2x faster, slightly lower F1
    max_length       = 64                # GoEmotions comments are short; 64 subword tokens covers almost all
    batch_size       = 32
    epochs           = 4
    lr_encoder       = 2e-5
    lr_head          = 1e-3
    weight_decay     = 0.01
    warmup_ratio     = 0.06
    dropout          = 0.1
    loss_type        = "bce_posweight"   # "bce" | "bce_posweight" | "focal"
    pos_weight_clip  = 10.0
    focal_gamma      = 2.0
    select_threshold = 0.3               # fixed threshold used ONLY to pick the best epoch
    patience         = 2
    out_dir          = "artifacts_text"

os.makedirs(CFG.out_dir, exist_ok=True)

## 1. Data and tokenization

In [ ]:
ds = load_dataset("google-research-datasets/go_emotions", "simplified")

label_names = ds["train"].features["labels"].feature.names
NUM_LABELS = len(label_names)
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

print(ds)
print("Number of emotions:", NUM_LABELS)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

def multi_hot(labels):
    v = np.zeros(NUM_LABELS, dtype=np.float32)
    v[labels] = 1.0
    return v.tolist()

def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=CFG.max_length)
    enc["labels"] = [multi_hot(l) for l in batch["labels"]]
    return enc

test_texts = list(ds["test"]["text"])      # kept for error analysis
tok_ds = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)

lens = [len(x) for x in tok_ds["train"]["input_ids"]]
print("Token length  mean: %.1f | p95: %d | p99: %d | max: %d" % (
    np.mean(lens), np.percentile(lens, 95), np.percentile(lens, 99), max(lens)))
print("Truncated at max_length: %.2f%%" % (100 * np.mean([l >= CFG.max_length for l in lens])))

In [ ]:
collator = DataCollatorWithPadding(tokenizer)   # pads each batch to its own longest sequence

def make_loader(split, shuffle=False, batch_size=None):
    return DataLoader(
        tok_ds[split],
        batch_size=batch_size or CFG.batch_size,
        shuffle=shuffle,
        collate_fn=collator,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

train_loader = make_loader("train", shuffle=True)
val_loader   = make_loader("validation", batch_size=128)
test_loader  = make_loader("test", batch_size=128)

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})

## 2. Model

Mean pooling over non-padding tokens, then dropout and a linear head. `forward(..., return_embedding=True)` also returns the pooled vector, which is what the multimodal fusion model consumes.

In [ ]:
class EmotionTransformer(nn.Module):
    def __init__(self, model_name_or_dir, num_labels, dropout=0.1, pretrained=True):
        super().__init__()
        if pretrained:
            self.encoder = AutoModel.from_pretrained(model_name_or_dir)
        else:   # build the architecture only; weights come from a saved state_dict
            self.encoder = AutoModel.from_config(AutoConfig.from_pretrained(model_name_or_dir))
        self.hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

    @staticmethod
    def mean_pool(last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).to(last_hidden.dtype)
        return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask, return_embedding=False):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        emb = self.mean_pool(out.last_hidden_state, attention_mask)
        logits = self.classifier(self.dropout(emb))
        return (logits, emb) if return_embedding else logits

model = EmotionTransformer(CFG.model_name, NUM_LABELS, CFG.dropout).to(device)
print("Parameters: %.1fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

## 3. Loss

GoEmotions is heavily imbalanced (see your EDA: `neutral`/`admiration` vs `grief`/`pride`). `bce_posweight` uses `sqrt(neg/pos)` per class, clipped, which boosts rare emotions without the instability of the raw ratio.

In [ ]:
class FocalBCE(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        return (((1 - p_t) ** self.gamma) * bce).mean()

def build_criterion():
    if CFG.loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    if CFG.loss_type == "focal":
        return FocalBCE(CFG.focal_gamma)
    if CFG.loss_type == "bce_posweight":
        y = np.array(tok_ds["train"]["labels"], dtype=np.float32)
        pos = y.sum(0); neg = len(y) - pos
        w = np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1.0, CFG.pos_weight_clip)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(w, dtype=torch.float32, device=device))
    raise ValueError(CFG.loss_type)

criterion = build_criterion()
print("Loss:", CFG.loss_type)

## 4. Training

In [ ]:
no_decay = ("bias", "LayerNorm.weight")
enc_named = list(model.encoder.named_parameters())
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in enc_named if not any(nd in n for nd in no_decay)],
     "lr": CFG.lr_encoder, "weight_decay": CFG.weight_decay},
    {"params": [p for n, p in enc_named if any(nd in n for nd in no_decay)],
     "lr": CFG.lr_encoder, "weight_decay": 0.0},
    {"params": model.classifier.parameters(), "lr": CFG.lr_head, "weight_decay": 0.0},
])

total_steps = len(train_loader) * CFG.epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(CFG.warmup_ratio * total_steps), total_steps
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print("Total steps:", total_steps, "| AMP:", use_amp)

In [ ]:
def train_one_epoch():
    model.train()
    total = 0.0
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device).float()

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(ids, mask)
        loss = criterion(logits.float(), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
    return total / len(train_loader)

@torch.no_grad()
def collect(loader):
    # returns probabilities, true labels, mean loss
    model.eval()
    probs, labs, total = [], [], 0.0
    for batch in loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device).float()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(ids, mask)
        total += criterion(logits.float(), labels).item()
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        labs.append(labels.cpu().numpy())
    return np.vstack(probs), np.vstack(labs), total / len(loader)

def quick_metrics(labels, preds):
    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
        "recall_micro": recall_score(labels, preds, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
    }

In [ ]:
history, best_f1, bad_epochs = [], -1.0, 0
ckpt_path = os.path.join(CFG.out_dir, "model.pt")

for epoch in range(1, CFG.epochs + 1):
    t0 = time.time()
    train_loss = train_one_epoch()
    vp, vl, val_loss = collect(val_loader)
    m = quick_metrics(vl, (vp >= CFG.select_threshold).astype(int))
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, **m})

    print(f"Epoch {epoch}/{CFG.epochs} | {time.time()-t0:.0f}s | train {train_loss:.4f} | "
          f"val {val_loss:.4f} | micro-F1 {m['micro_f1']:.4f} | macro-F1 {m['macro_f1']:.4f}")

    if m["macro_f1"] > best_f1:
        best_f1, bad_epochs = m["macro_f1"], 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        bad_epochs += 1
        if bad_epochs >= CFG.patience:
            print("Early stopping.")
            break

history_df = pd.DataFrame(history)
history_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
ax[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history_df["epoch"], history_df["micro_f1"], label="micro F1")
ax[1].plot(history_df["epoch"], history_df["macro_f1"], label="macro F1")
ax[1].set_title(f"Validation F1 (threshold {CFG.select_threshold})"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Threshold tuning (validation only)

- One **global** threshold maximising micro-F1.
- One **per-emotion** threshold maximising that emotion's F1.
- Emotions with fewer than `min_support` validation positives (e.g. `grief`, `pride`, `relief`) are too small to tune reliably, so they fall back to the global threshold instead of overfitting to a handful of examples.

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))

val_probs, val_labels, _ = collect(val_loader)
test_probs, test_labels, _ = collect(test_loader)

GRID = np.arange(0.05, 0.91, 0.01)

def f1_over_grid(p_col, y_col, grid):
    preds = p_col[:, None] >= grid[None, :]
    pos = (y_col[:, None] == 1)
    tp = (preds & pos).sum(0); fp = (preds & ~pos).sum(0); fn = (~preds & pos).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def tune_thresholds(probs, labels, grid=GRID, min_support=20):
    preds = probs[:, :, None] >= grid[None, None, :]
    pos = (labels[:, :, None] == 1)
    tp = (preds & pos).sum((0, 1)); fp = (preds & ~pos).sum((0, 1)); fn = (~preds & pos).sum((0, 1))
    micro = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    t_global = float(grid[micro.argmax()])

    thr = np.full(probs.shape[1], t_global)
    support = labels.sum(0)
    for c in range(probs.shape[1]):
        if support[c] >= min_support:
            thr[c] = grid[f1_over_grid(probs[:, c], labels[:, c], grid).argmax()]
    return thr, t_global

best_thr, t_global = tune_thresholds(val_probs, val_labels)
print("Global threshold:", round(t_global, 2))

pd.DataFrame({
    "emotion": label_names,
    "val_support": val_labels.sum(0).astype(int),
    "threshold": best_thr.round(2),
})

## 6. Evaluation on the test set

In [ ]:
comparison = pd.DataFrame({
    "fixed_0.5":        quick_metrics(test_labels, (test_probs >= 0.5).astype(int)),
    "global_tuned":     quick_metrics(test_labels, (test_probs >= t_global).astype(int)),
    "per_emotion_tuned": quick_metrics(test_labels, (test_probs >= best_thr).astype(int)),
})
comparison

In [ ]:
preds_tuned = (test_probs >= best_thr).astype(int)
print(classification_report(test_labels, preds_tuned, target_names=label_names, zero_division=0))

f1_df = pd.DataFrame({
    "emotion": label_names,
    "f1": f1_score(test_labels, preds_tuned, average=None, zero_division=0),
    "support": test_labels.sum(0).astype(int),
}).sort_values("f1", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=f1_df, x="f1", y="emotion")
plt.title(f"Per-emotion F1 on test ({CFG.model_name}, tuned thresholds)")
plt.show()
f1_df

### Ekman-7 view

Most audio/video emotion datasets (e.g. MELD, RAVDESS) use basic emotions, not 28 fine-grained ones. The standard GoEmotions-to-Ekman grouping lets the text branch output labels that line up with the other modalities. A group's probability is the max over its member emotions.

In [ ]:
EKMAN = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude", "love",
                 "optimism", "relief", "pride", "admiration", "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"],
}
ekman_names = list(EKMAN)
ekman_idx = {g: [label_names.index(l) for l in ls] for g, ls in EKMAN.items()}
assert sorted(i for v in ekman_idx.values() for i in v) == list(range(NUM_LABELS)), "mapping must cover all 28 labels once"

def to_ekman(mat):
    return np.stack([mat[:, idx].max(1) for idx in ekman_idx.values()], axis=1)

val_e_probs, val_e_labels = to_ekman(val_probs), to_ekman(val_labels)
test_e_probs, test_e_labels = to_ekman(test_probs), to_ekman(test_labels)

ekman_thr, ekman_global = tune_thresholds(val_e_probs, val_e_labels)
test_e_preds = (test_e_probs >= ekman_thr).astype(int)

print(classification_report(test_e_labels, test_e_preds, target_names=ekman_names, zero_division=0))

### Error analysis

In [ ]:
errors = (preds_tuned != test_labels).sum(1)
worst = np.argsort(-errors)[:12]

for i in worst:
    true = [label_names[j] for j in np.where(test_labels[i] == 1)[0]]
    pred = [label_names[j] for j in np.where(preds_tuned[i] == 1)[0]]
    print("TEXT:", test_texts[i])
    print("TRUE:", true, "| PRED:", pred)
    print("-" * 80)

## 7. Save artifacts and a reusable predictor

Everything needed for inference (weights, tokenizer, encoder config, thresholds, label order, Ekman map) goes into one folder, so the predictor works without re-downloading the pretrained model.

In [ ]:
tokenizer.save_pretrained(CFG.out_dir)
model.encoder.config.save_pretrained(CFG.out_dir)       # writes config.json (encoder architecture)
# model.pt (best weights) was already written during training

meta = {
    "model_name": CFG.model_name,
    "max_length": CFG.max_length,
    "embedding_dim": model.hidden_size,
    "label_names": label_names,
    "thresholds": [float(t) for t in best_thr],
    "global_threshold": t_global,
    "ekman_map": EKMAN,
    "ekman_thresholds": [float(t) for t in ekman_thr],
}
with open(os.path.join(CFG.out_dir, "emotion_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(sorted(os.listdir(CFG.out_dir)))

In [ ]:
class TextEmotionPredictor:
    def __init__(self, artifact_dir, device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        with open(os.path.join(artifact_dir, "emotion_meta.json")) as f:
            meta = json.load(f)
        self.labels = meta["label_names"]
        self.thr = np.array(meta["thresholds"])
        self.ekman_thr = np.array(meta["ekman_thresholds"])
        self.max_length = meta["max_length"]
        self.ekman_idx = {g: [self.labels.index(l) for l in ls] for g, ls in meta["ekman_map"].items()}

        self.tokenizer = AutoTokenizer.from_pretrained(artifact_dir)
        self.model = EmotionTransformer(artifact_dir, len(self.labels), pretrained=False)
        self.model.load_state_dict(torch.load(os.path.join(artifact_dir, "model.pt"), map_location=self.device))
        self.model.to(self.device).eval()

    def _ekman(self, probs):
        return np.stack([probs[:, idx].max(1) for idx in self.ekman_idx.values()], axis=1)

    @torch.no_grad()
    def encode(self, texts, batch_size=64):
        # returns (probs [N, 28], embeddings [N, hidden]) - this is what the fusion model uses
        if isinstance(texts, str):
            texts = [texts]
        probs, embs = [], []
        for i in range(0, len(texts), batch_size):
            enc = self.tokenizer(texts[i:i + batch_size], truncation=True, max_length=self.max_length,
                                 padding=True, return_tensors="pt").to(self.device)
            logits, emb = self.model(enc["input_ids"], enc["attention_mask"], return_embedding=True)
            probs.append(torch.sigmoid(logits.float()).cpu().numpy())
            embs.append(emb.float().cpu().numpy())
        return np.vstack(probs), np.vstack(embs)

    def predict(self, texts, top_k=None):
        single = isinstance(texts, str)
        texts = [texts] if single else list(texts)
        probs, _ = self.encode(texts)
        e_probs = self._ekman(probs)
        out = []
        for i, text in enumerate(texts):
            hits = [(self.labels[j], float(probs[i, j])) for j in range(len(self.labels)) if probs[i, j] >= self.thr[j]]
            hits.sort(key=lambda x: x[1], reverse=True)
            if not hits:                                   # nothing crossed its threshold -> report the top-1
                j = int(probs[i].argmax())
                hits = [(self.labels[j], float(probs[i, j]))]
            ekman = {g: float(e_probs[i, k]) for k, g in enumerate(self.ekman_idx)}
            out.append({
                "text": text,
                "emotions": hits[:top_k] if top_k else hits,
                "ekman": ekman,
                "ekman_top": max(ekman, key=ekman.get),
            })
        return out[0] if single else out

predictor = TextEmotionPredictor(CFG.out_dir)

for s in [
    "I can't believe I finally got the job, thank you so much!",
    "This is so annoying, why does nobody listen to me?",
    "I miss her so much, it still hurts.",
    "Wait, what? You did that without telling anyone?",
]:
    r = predictor.predict(s)
    print(s)
    print("  emotions:", [(e, round(p, 2)) for e, p in r["emotions"]])
    print("  ekman top:", r["ekman_top"], "\n")

## 8. Hooking the text branch into the multimodal system

Per utterance, the text branch exports three things: 28-d emotion probabilities, 7-d Ekman probabilities, and the 768-d embedding. The other branches would export the same kind of vector:

| Branch | Typical encoder | Output |
|---|---|---|
| Text | this notebook (RoBERTa) | 768-d embedding + probabilities |
| Audio | Wav2Vec2 / HuBERT (mean-pooled over time) | 768-d embedding |
| Video | face crops per frame -> CNN/ViT, pooled over frames | 512-d embedding |

`MultimodalEmotionFusion` below is a small gated-attention fusion head. It takes whichever modalities are present (a missing face or silent clip is common in practice) and trains with random modality dropout so it stays robust when one is absent. The audio/video tensors here are random placeholders, only to prove the shapes work.

In [ ]:
def extract_text_features(predictor, texts, path=None):
    probs, emb = predictor.encode(texts)
    feats = {"probs": probs, "ekman": predictor._ekman(probs), "embedding": emb}
    if path:
        np.savez_compressed(path, **feats)
    return feats

demo = extract_text_features(predictor, ["I love this!", "I am so tired of all this."])
print({k: v.shape for k, v in demo.items()})

In [ ]:
class MultimodalEmotionFusion(nn.Module):
    def __init__(self, dims, hidden=256, num_classes=7, dropout=0.3, modality_dropout=0.15):
        super().__init__()
        self.names = list(dims)
        self.modality_dropout = modality_dropout
        self.proj = nn.ModuleDict({
            m: nn.Sequential(nn.Linear(d, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout))
            for m, d in dims.items()
        })
        self.gate = nn.Linear(hidden, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, num_classes)
        )

    def forward(self, feats, present=None):
        # feats: {name: [B, D]}; present: {name: [B] bool} (True if that modality exists for the sample)
        h = torch.stack([self.proj[m](feats[m]) for m in self.names], dim=1)          # [B, M, H]
        B = h.size(0)
        if present is None:
            mask = torch.ones(B, len(self.names), dtype=torch.bool, device=h.device)
        else:
            mask = torch.stack([present[m] for m in self.names], dim=1)

        if self.training and self.modality_dropout > 0:
            keep = mask & (torch.rand(mask.shape, device=h.device) >= self.modality_dropout)
            empty = ~keep.any(dim=1)
            keep[empty] = mask[empty]          # never drop every modality for a sample
            mask = keep

        scores = self.gate(h).squeeze(-1).masked_fill(~mask, float("-inf"))
        w = scores.softmax(dim=1)                                                   # modality weights
        fused = (w.unsqueeze(-1) * h).sum(1)
        return self.classifier(fused), w

fusion = MultimodalEmotionFusion({"text": 768, "audio": 768, "video": 512}).to(device)
B = 4
feats = {"text": torch.randn(B, 768, device=device),
         "audio": torch.randn(B, 768, device=device),
         "video": torch.randn(B, 512, device=device)}
present = {"text": torch.ones(B, dtype=torch.bool, device=device),
           "audio": torch.ones(B, dtype=torch.bool, device=device),
           "video": torch.tensor([True, False, True, False], device=device)}   # two samples have no video

fusion.eval()
logits, w = fusion(feats, present)
print("logits:", tuple(logits.shape), "| modality weights:\n", w.detach().cpu().numpy().round(2))

## What to do next

1. **Fine-tune on dialogue data.** GoEmotions is Reddit text; a conversational multimodal dataset such as MELD (text + audio + video, 7 emotions that match the Ekman grouping) will shift the text branch toward spoken-style utterances. Start from `artifacts_text/model.pt` and train on MELD with the Ekman-7 head.
2. **Compare against the BiLSTM.** Paste your BiLSTM tuned-threshold numbers next to the `comparison` table above so the report shows the gain.
3. **Ablations worth reporting:** `loss_type` (bce vs posweight vs focal), `distilroberta-base` vs `roberta-base`, and multi-seed averaging.
4. **Calibration:** per-emotion temperature scaling on validation makes the probabilities more trustworthy as fusion inputs.
5. **Deployment:** distil to a smaller encoder or export to ONNX if the full system needs to run in real time.